In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

np.random.seed(42)
torch.manual_seed(42)

In [ ]:
# Let's load airline passengers dataset
import pandas as pd

url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
df = pd.read_csv(url)
df.columns = ['Month', 'Passengers']
df['Month'] = pd.to_datetime(df['Month'])

print(df[['Month', 'Passengers']].head(150))
print('Airline Passengers Dataset size:', len(df))

# Convert to numpy array
data = df['Passengers'].values.astype(float)

context_length = 100  # Use 100 months for context
prediction_length = 24  # Predict next 24 months (2 years)

context = data[:context_length]
target = data[context_length:context_length + prediction_length]

plt.figure(figsize=(12, 6))
plt.plot(range(len(context)), context, label='Context (Historical Data)', color='blue')
plt.plot(range(len(context), len(context) + len(target)), target, label='Target (Future Data)', color='red')
plt.axvline(x=len(context), color='red', linestyle='--', linewidth=2, label='Forecast Start')
plt.xlabel('Months')
plt.ylabel('Number of Passengers')
plt.grid(True, alpha=0.3)
plt.legend()
plt.title('Airline Passengers Data')

In [ ]:
!pip install -q chronos-forecasting

In [ ]:
# Let load the Chronos Tiny T5 model

from chronos import ChronosPipeline

pipeline = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-tiny",
    device_map="cuda",  # Use "mps" if you have a Apple Silicon
    torch_dtype=torch.float32,
)

In [ ]:
# Let's visualize the forecast

context_tensor = torch.tensor(context, dtype=torch.float32).unsqueeze(0)

forecast = pipeline.predict(
    inputs=context_tensor,
    prediction_length=prediction_length,
    num_samples=100,  # Generate 100 possible futures
)

# forecase_samples is of shape (batch_size, num_samples, prediction_length).
forecast_samples = forecast[0].numpy()
# We got the median, p90 and p10 percentiles across 100 samples
forecast_median = np.median(forecast_samples, axis=0)
forecast_10p = np.percentile(forecast_samples, 10, axis=0)
forecast_90p = np.percentile(forecast_samples, 90, axis=0)

plt.figure(figsize=(14, 6))
plt.plot(range(len(context)), context, label='Context (Historical Data)', color='blue')
plt.plot(range(len(context), len(context) + len(target)), target, label='Actual Target (Future Data)', color='red')
plt.plot(range(len(context), len(context) + len(target)), forecast_median, label='Forecast Median', color='green')
plt.fill_between(
    range(len(context), len(context) + len(target)),
    forecast_10p,
    forecast_90p,
    color='green',
    alpha=0.3,
    label='80% Prediction Interval (10th-90th Percentile)'
)
plt.axvline(x=len(context), color='red', linestyle='--', linewidth=2, label='Forecast Start')
plt.xlabel('Months')
plt.ylabel('Number of Passengers')
plt.title('Airline Passengers Forecast with Chronos Tiny T5')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

# Calculate error metrics
mae = np.mean(np.abs(target - forecast_median))
mape = np.mean(np.abs((target - forecast_median) / target)) * 100
rmse = np.sqrt(np.mean((target - forecast_median)**2))

print(f"Forecast Accuracy:")
print(f"- MAE (Mean Absolute Error): {mae:.2f} passengers")
print(f"- MAPE (Mean Absolute Percentage Error): {mape:.2f}%")
print(f"- RMSE (Root Mean Squared Error): {rmse:.2f} passengers")

In [ ]:
# Let's visualize the binning process

# Let's use those ceters as examples: [-2. -1.  0.  1.  2.]
centers = np.linspace(-2, 2, 5)
# And following boundaries: [-inf -1.5 -0.5  0.5  1.5  inf]
boundaries = np.concatenate([[-np.inf], (centers[1:] + centers[:-1]) / 2, [np.inf]])

# We generate 1000 values between -2 and 2
values = np.linspace(-2, 2, 1000)
bins = []

# For each value we look up the corresponding bin
for v in values:
    d = np.digitize(v, boundaries) - 1
    c = np.clip(d, 0, len(centers) - 1)
    bins.append(c)

# Convert bins to corresponding center values.
# Important: this is where we lose precision because multiple original values may map to the same center.
converted = centers[bins]

fig, axes = plt.subplots(1, 1, figsize=(14, 4))

# Plot original values and converted values
axes.plot(values, values, linewidth=3, color='blue')
axes.set_title('Values:', fontsize=14, fontweight='bold')
axes.set_xlabel('x')
axes.set_ylabel('f(x) = x')
axes.grid(True, alpha=0.3)
axes.plot(values, converted, linewidth=2, color='red')
axes.set_title('Assign to Bins:', fontsize=14, fontweight='bold')
axes.set_xlabel('Values')
axes.set_ylabel('Bin ID')
axes.grid(True, alpha=0.3)

# Mark boundaries
for i, boundary in enumerate(boundaries[1:-1]):
    axes.axvline(x=boundary, color='red', linestyle=':', alpha=0.5, linewidth=2)
    axes.text(boundary, 4.3, f'{boundary:.1f}', fontsize=9, ha='center', color='red')

# Mark bin centers
for center in centers:
    axes.scatter([center], [center], color='green', s=100, zorder=5, marker='o')

In [ ]:
# Let's implement ChronosTokenizer class (compatible with ChronosDataset interface)

class ChronosTokenizer:
    def __init__(self, low_limit=-15.0, high_limit=15.0, config=None):
        self.config = config
        self.low_limit = low_limit
        self.high_limit = high_limit
        # Number of bins for actual values (excluding special tokens)
        n_value_tokens = config.n_tokens - config.n_special_tokens - 1
        # Create bin centers and boundaries
        self.centers = torch.linspace(low_limit, high_limit, n_value_tokens)
        self.boundaries = torch.cat([
            torch.tensor([-1e20]),
            (self.centers[1:] + self.centers[:-1]) / 2,
            torch.tensor([1e20])
        ])

    def _input_transform(self, context, scale=None):
        context = context.to(dtype=torch.float32)
        attention_mask = ~torch.isnan(context)
        # Compute scale if not provided (mean absolute value)
        if scale is None:
            scale = torch.nansum(torch.abs(context) * attention_mask, dim=-1) / torch.nansum(attention_mask, dim=-1)
            scale[~(scale > 0)] = 1.0
        # Normalize by scale
        scaled_context = context / scale.unsqueeze(dim=-1)
        # Convert to token IDs using bucketize
        token_ids = (
            torch.bucketize(scaled_context, self.boundaries, right=True)
            + self.config.n_special_tokens  # Offset by special tokens
        )
        # Clamp to valid range
        token_ids = token_ids.clamp(0, self.config.n_tokens - 1)
        # Set padding tokens for NaN values
        token_ids[~attention_mask] = self.config.pad_token_id
        return token_ids, attention_mask, scale

    def _append_eos_token(self, token_ids, attention_mask):
        """Append EOS token to sequences."""
        batch_size = token_ids.shape[0]
        eos_tokens = torch.full((batch_size, 1), fill_value=self.config.eos_token_id)
        token_ids = torch.cat([token_ids, eos_tokens], dim=1)
        eos_mask = torch.full((batch_size, 1), fill_value=True)
        attention_mask = torch.cat([attention_mask, eos_mask], dim=1)
        return token_ids, attention_mask

    def context_input_transform(self, context):
        # Truncate to context_length if needed
        if context.shape[-1] > self.config.context_length:
            context = context[..., -self.config.context_length:]
        token_ids, attention_mask, scale = self._input_transform(context)
        # Append EOS token for seq2seq models
        if self.config.use_eos_token and self.config.model_type == "seq2seq":
            token_ids, attention_mask = self._append_eos_token(token_ids, attention_mask)
        return token_ids, attention_mask, scale

    def label_input_transform(self, label, scale):
        token_ids, attention_mask, _ = self._input_transform(label, scale=scale)
        if self.config.use_eos_token:
            token_ids, attention_mask = self._append_eos_token(token_ids, attention_mask)
        return token_ids, attention_mask

    def output_transform(self, samples, scale):
        scale_unsqueezed = scale.unsqueeze(-1).unsqueeze(-1)
        # Convert token IDs to bin indices (subtract special token offset)
        indices = torch.clamp(
            samples - self.config.n_special_tokens - 1,
            min=0,
            max=len(self.centers) - 1
        )
        # Look up bin centers and multiply by scale
        return self.centers[indices] * scale_unsqueezed

In [ ]:
# Let's test the ChronosTokenizer class (now compatible with ChronosDataset)

import torch
import numpy as np
import matplotlib.pyplot as plt

# Create a config for testing
class DummyConfig:
    n_tokens = 100
    n_special_tokens = 2
    pad_token_id = 0
    eos_token_id = 1
    use_eos_token = True
    model_type = "seq2seq"
    context_length = 100
    prediction_length = 5

test_config = DummyConfig()
test_tokenizer = ChronosTokenizer(low_limit=-2, high_limit=2, config=test_config)

# Sine wave visualization
t = np.linspace(0, 4 * np.pi, 100)
sine_wave = 10 + 5 * np.sin(t) + np.random.normal(0, 0.5, len(t))
sine_tensor = torch.tensor(sine_wave, dtype=torch.float32).unsqueeze(0)
sine_token_ids, _, sine_scale = test_tokenizer.context_input_transform(sine_tensor)
sine_samples = sine_token_ids.unsqueeze(1)
sine_reconstructed = test_tokenizer.output_transform(sine_samples, sine_scale).squeeze().numpy()

# Ensure the lengths match for plotting
plot_len = min(len(t), len(sine_reconstructed))

fig, axes = plt.subplots(2, 1, figsize=(12, 8))
axes[0].plot(t, sine_wave, label='Original Wave', color='blue')
axes[0].set_title('Original Wave')
axes[0].grid(True, alpha=0.3)

axes[1].plot(t[:plot_len], sine_reconstructed[:plot_len], label='Reconstructed from Tokens', color='orange')
axes[1].set_title('Reconstructed Wave from Tokens (New Interface)')
axes[1].grid(True, alpha=0.3)

for ax in axes:
    ax.set_xlabel('Time')
    ax.set_ylabel('Amplitude')
    ax.legend()


In [ ]:
import torch
import torch.nn as nn
from torch.nn import TransformerEncoder, TransformerEncoderLayer, TransformerDecoder, TransformerDecoderLayer

In [ ]:
# Implement RoPE (Rotary Position Embeddings)

class RotaryPositionalEmbedding(nn.Module):

    def __init__(self, d_model, max_seq_len=128, base=10000):
        super().__init__()
        self.d_model = d_model
        self.max_seq_len = max_seq_len
        self.base = base
        # Precompute frequency bands (tensor([1.0000, 0.1000, 0.0100, 0.0010]) for d_model=8)
        inv_freq = 1.0 / (base ** (torch.arange(0, d_model, 2).float() / d_model))
        self.register_buffer('inv_freq', inv_freq)
        # Cache for efficiency
        self._seq_len_cached = None
        self._cos_cached = None
        self._sin_cached = None

    def _update_cache(self, seq_len, device):
        """Update cached cos/sin values if sequence length changes."""
        if seq_len != self._seq_len_cached:
            self._seq_len_cached = seq_len
            # Position indices ([0., 1., 2., 3., 4., 5., 6., 7.] for seq_len=8)
            t = torch.arange(seq_len, device=device).type_as(self.inv_freq)
            # Compute frequencies for each position
            freqs = torch.outer(t, self.inv_freq)  # (seq_len, d_model//2)
            # Duplicate for full dimensionality
            emb = torch.cat([freqs, freqs], dim=-1)  # (seq_len, d_model)
            self._cos_cached = emb.cos()[None, :, :]  # (1, seq_len, d_model)
            self._sin_cached = emb.sin()[None, :, :]  # (1, seq_len, d_model)


    def rotate_half(self, x):
        """Rotate half the hidden dims of the input."""
        # For a single pair of coordinates (x1, x2), rotating them by 'theta' angle
        # is defined by the standard 2D rotation matrix.
        x1, x2 = x.chunk(2, dim=-1)
        return torch.cat([-x2, x1], dim=-1)

    def apply_rotary_pos_emb(self, x, cos, sin):
        """Apply rotary position embedding to input tensor."""
        # x1N = x1 * cos - x2 * sin
        # x2N = x2 * cos + x1 * sin
        return (x * cos) + (self.rotate_half(x) * sin)

    def forward(self, x):
        _, seq_len, _ = x.shape
        self._update_cache(seq_len, x.device)
        return self.apply_rotary_pos_emb(
            x,
            self._cos_cached[:, :seq_len, :],
            self._sin_cached[:, :seq_len, :]
        )

In [ ]:
# Test RoPE with three words represented as 2D vectors
from matplotlib.patches import Patch

rope = RotaryPositionalEmbedding(d_model=2, max_seq_len=4)
test_input = torch.tensor([[[1.0, 0.0],
                            [0.7, 0.7],
                            [0.0, 1.0]]])

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
def plot_2d_rope_vectors(index, data):
    ax[index].quiver(
        [0, 0, 0], [0, 0, 0],
        data[0, :, 0].numpy(), data[0, :, 1].numpy(),
        angles='xy', scale_units='xy', scale=1, color=['r', 'g', 'b']
    )
    ax[index].set_xlim(-1.5, 1.5)
    ax[index].set_ylim(-1.5, 1.5)
    ax[index].set_title('Before RoPE')
    ax[index].grid(True)

legend_elements = [
    Patch(facecolor='r', label='Word 0 (The)'),
    Patch(facecolor='g', label='Word 1 (cat)'),
    Patch(facecolor='b', label='Word 2 (sat)')
]
ax[0].legend(handles=legend_elements, loc='upper left')
ax[1].legend(handles=legend_elements, loc='upper left')

# Plot the three test_input vectors before and after RoPE
plot_2d_rope_vectors(0, test_input)
output = rope(test_input)
plot_2d_rope_vectors(1, output)

# Let's get the angles between the original and rotated vectors
# a * b = |a| * |b| * cos(θ) => θ = acos( (a * b) / (|a| * |b|) )
def angle_between(v1, v2):
    unit_v1 = v1 / torch.norm(v1)
    unit_v2 = v2 / torch.norm(v2)
    dot_product = torch.clamp(torch.dot(unit_v1, unit_v2), -1.0, 1.0)
    angle = torch.acos(dot_product)
    return angle

for i in range(test_input.shape[1]):
    original_vec = test_input[0, i]
    rotated_vec = output[0, i]
    angle = angle_between(original_vec, rotated_vec)
    print(f"Vector {i}: Angle between original and rotated = {angle.item() * (180.0 / np.pi):.2f} degrees")

In [ ]:
# Let's add one more dimension (4D vectors) to see fast and slow rotations

v1 = torch.tensor([1.0, 0.0, 0.0, 1.0])
v2 = torch.tensor([0.0, 1.0, 1.0, 0.0])
v3 = torch.tensor([0.7, 0.7, 0.7, 0.7])
test_input = torch.stack([v1, v2, v3]).unsqueeze(0)

rope = RotaryPositionalEmbedding(d_model=4, max_seq_len=3)

output = rope(test_input)

def extract_pairs(tensor):
    t = tensor.squeeze(0)
    p1 = torch.stack([t[:, 0], t[:, 2]], dim=1).detach().numpy()
    p2 = torch.stack([t[:, 1], t[:, 3]], dim=1).detach().numpy()
    return p1, p2

input_p1, input_p2 = extract_pairs(test_input)
output_p1, output_p2 = extract_pairs(output)

fig, axes = plt.subplots(2, 2, figsize=(12, 12))
colors = ['r', 'g', 'b']
labels = ['Word 0 (Red)', 'Word 1 (Green)', 'Word 2 (Blue)']

def plot_on_ax(ax, data, title):
    for i in range(3):
        u, v = data[i, 0], data[i, 1]
        ax.quiver(0, 0, u, v, angles='xy', scale_units='xy', scale=1,
                  color=colors[i], width=0.015, headwidth=5, alpha=0.9, label=labels[i])
    ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5); ax.set_aspect('equal')
    ax.grid(True, linestyle=':', alpha=0.6)
    ax.axhline(0, color='black', linewidth=0.5); ax.axvline(0, color='black', linewidth=0.5)
    ax.set_title(title, fontsize=11, fontweight='bold')
    circle = plt.Circle((0, 0), 1, color='lightgrey', fill=False, linestyle='--')
    ax.add_artist(circle)

plot_on_ax(axes[0, 0], input_p1, "1. Initial - Pair 1 (Dims 0 & 2)")
plot_on_ax(axes[0, 1], output_p1, "2. Rotated - Pair 1 (Fast Rotation)")

plot_on_ax(axes[1, 0], input_p2, "3. Initial - Pair 2 (Dims 1 & 3)")
plot_on_ax(axes[1, 1], output_p2, "4. Rotated - Pair 2 (Slow Rotation)")

handles = [Patch(facecolor=c, label=l) for c, l in zip(colors, labels)]
fig.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, 0.03), ncol=3)

plt.suptitle("RoPE Visualization", fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.subplots_adjust(bottom=0.1)
plt.show()

In [ ]:
# Let's understndand RoPE (deep dive)

import torch
import matplotlib.pyplot as plt
import numpy as np

def plot_interference():
    dim = 128
    max_distance = 200
    base = 10000.0

    # 1. Calculate frequencies based on: 10000 ^ (-2i/d)
    i = torch.arange(0, dim, 2).float()
    print("Indicies:", i[:5], "...", i[-5:])
    theta = 1.0 / (base ** (i / dim))
    print("Frequencies:", theta[:5], "...", theta[-5:])

    # 2. Calculate the Sum of Cosines for each distance k
    distances = torch.arange(max_distance).float()
    print("Positions:", distances[:5], "...", distances[-5:])

    # Outer product to get all angles: (distances x frequencies)
    angles = torch.outer(distances, theta)

    # This represents the "Attention Score"
    scores = torch.sum(torch.cos(angles), dim=1) / (dim / 2)
    print("Scores examples:", scores[:5], "...", scores[-5:])

    # 3. Plotting
    plt.figure(figsize=(12, 8))

    # --- Subplot 1: The Result (The Decay) ---
    plt.subplot(2, 1, 1)
    plt.plot(distances.numpy(), scores.numpy(), color='#2c3e50', linewidth=2)
    plt.title(f"The Result: Sum of {dim//2} Cosines (Constructive vs Destructive Interference)", fontsize=14)
    plt.ylabel("Net Attention Score", fontsize=12)
    plt.xlabel("Relative Distance (Tokens)", fontsize=12)
    plt.axhline(0, color='red', linestyle='--', alpha=0.5)
    plt.grid(True, alpha=0.3)

    # Annotations
    plt.text(2, 0.8, "Distance 0:\nEveryone pulls\ntogether (+1)", color='green', fontweight='bold')
    plt.text(50, 0.3, "Distance 50+:\nEveryone pulling\nrandom directions\n(Net ~ 0)", color='red')

    # --- Subplot 2: The Cause (Individual Waves) ---
    plt.subplot(2, 1, 2)

    # Pick specific dimension pairs to show the speed difference
    indices = [0, 10, 32, 63] # Fast, Medium, Slow, Very Slow
    labels = ["Fastest Pair (Dim 0)", "Medium Pair (Dim 10)", "Slow Pair (Dim 32)", "Slowest Pair (Dim 63)"]

    for idx, label in zip(indices, labels):
        freq = theta[idx].item()
        y = torch.cos(distances * freq).numpy()
        plt.plot(distances.numpy(), y, label=label, alpha=0.8, linewidth=1.5)

    plt.title("The Cause: Individual Dimensions Going Out of Sync", fontsize=14)
    plt.ylabel("Cosine Value", fontsize=12)
    plt.xlabel("Relative Distance (Tokens)", fontsize=12)
    plt.legend(loc='upper right')
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('rope_decay_interference.png')
    print("Plot generated: rope_decay_interference.png")

plot_interference()

In [ ]:
# ============================================================================
# PROPER RoPE: Apply rotation to Q and K INSIDE each attention layer
# This is how LLaMA, Mistral, and other modern LLMs implement RoPE
# ============================================================================

class RoPEMultiHeadAttention(nn.Module):
    """Multi-head attention with RoPE applied to Q and K inside attention."""

    def __init__(self, d_model, n_heads, dropout=0.1, max_seq_len=512):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        assert self.head_dim * n_heads == d_model, "d_model must be divisible by n_heads"

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

        # RoPE for this attention layer
        self.rope = RotaryPositionalEmbedding(self.head_dim, max_seq_len=max_seq_len)

    def forward(self, x, attn_mask=None, key_padding_mask=None):
        batch_size, seq_len, _ = x.shape

        # Project to Q, K, V
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        # Reshape: (batch, seq, d_model) -> (batch, seq, n_heads, head_dim)
        q = q.view(batch_size, seq_len, self.n_heads, self.head_dim)
        k = k.view(batch_size, seq_len, self.n_heads, self.head_dim)
        v = v.view(batch_size, seq_len, self.n_heads, self.head_dim)

        # Apply RoPE to Q and K (this is the key difference!)
        # Reshape for RoPE: (batch, seq, n_heads, head_dim) -> (batch * n_heads, seq, head_dim)
        q = q.permute(0, 2, 1, 3).reshape(batch_size * self.n_heads, seq_len, self.head_dim)
        k = k.permute(0, 2, 1, 3).reshape(batch_size * self.n_heads, seq_len, self.head_dim)

        q = self.rope(q)  # Apply RoPE to Q
        k = self.rope(k)  # Apply RoPE to K

        # Reshape back: (batch * n_heads, seq, head_dim) -> (batch, n_heads, seq, head_dim)
        q = q.view(batch_size, self.n_heads, seq_len, self.head_dim)
        k = k.view(batch_size, self.n_heads, seq_len, self.head_dim)
        v = v.permute(0, 2, 1, 3)  # (batch, n_heads, seq, head_dim)

        # Scaled dot-product attention
        scale = self.head_dim ** -0.5
        attn_weights = torch.matmul(q, k.transpose(-2, -1)) * scale

        # Apply masks
        if attn_mask is not None:
            attn_weights = attn_weights + attn_mask
        if key_padding_mask is not None:
            attn_weights = attn_weights.masked_fill(
                key_padding_mask.unsqueeze(1).unsqueeze(2), float('-inf')
            )

        attn_weights = torch.softmax(attn_weights, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Apply attention to values
        attn_output = torch.matmul(attn_weights, v)

        # Reshape back: (batch, n_heads, seq, head_dim) -> (batch, seq, d_model)
        attn_output = attn_output.permute(0, 2, 1, 3).reshape(batch_size, seq_len, self.d_model)

        return self.out_proj(attn_output)


class RoPEEncoderLayer(nn.Module):
    """Transformer encoder layer with proper RoPE in attention."""

    def __init__(self, d_model, n_heads, dim_feedforward=2048, dropout=0.1, max_seq_len=512):
        super().__init__()
        self.self_attn = RoPEMultiHeadAttention(d_model, n_heads, dropout, max_seq_len)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, src, src_mask=None, src_key_padding_mask=None):
        # Self-attention with RoPE
        src2 = self.self_attn(src, attn_mask=src_mask, key_padding_mask=src_key_padding_mask)
        src = src + self.dropout1(src2)
        src = self.norm1(src)

        # Feedforward
        src2 = self.linear2(self.dropout(torch.relu(self.linear1(src))))
        src = src + self.dropout2(src2)
        src = self.norm2(src)

        return src


class ChronosTransformerEncoderRoPE(nn.Module):
    """Encoder with PROPER RoPE - rotation applied inside each attention layer."""

    def __init__(self,
                 embedding_size=4096,
                 d_model=256,
                 n_heads=4,
                 n_encoder_layers=4,
                 dim_feedforward=1024,
                 dropout=0.1,
                 context_length=512):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.n_encoder_layers = n_encoder_layers
        self.dim_feedforward = dim_feedforward
        self.dropout = dropout
        self.context_length = context_length
        self.token_embedding = nn.Embedding(embedding_size, d_model)

        # Stack of encoder layers with RoPE inside each attention
        self.layers = nn.ModuleList([
            RoPEEncoderLayer(d_model, n_heads, dim_feedforward, dropout, context_length)
            for _ in range(n_encoder_layers)
        ])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, src_tokens, src_padding_mask=None):
        # Token embedding (no positional encoding added - RoPE is inside attention!)
        x = self.token_embedding(src_tokens)

        # Pass through encoder layers (each applies RoPE to Q and K)
        for layer in self.layers:
            x = layer(x, src_key_padding_mask=src_padding_mask)

        return self.norm(x)

# Test ChronosTransformerEncoderRoPE
test_encoder = ChronosTransformerEncoderRoPE(
    embedding_size=4096,
    d_model=256,
    n_heads=4,
    n_encoder_layers=2,
    dim_feedforward=1024,
    dropout=0.1,
    context_length=128
)
test_input_tokens = torch.randint(0, 4096, (2, 128))  # Batch size 2, sequence length 128
test_encoder_output = test_encoder(test_input_tokens)
print(f"Input Tokens shape: {test_input_tokens.shape}")
print(f"Encoder Output shape: {test_encoder_output.shape}")

In [ ]:
class RoPECrossAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, memory, key_padding_mask=None):
        batch_size, tgt_len, _ = x.shape
        _, src_len, _ = memory.shape
        q = self.q_proj(x)
        k = self.k_proj(memory)
        v = self.v_proj(memory)
        # Reshape for multi-head attention
        q = q.view(batch_size, tgt_len, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        k = k.view(batch_size, src_len, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        v = v.view(batch_size, src_len, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        # Attention
        scale = self.head_dim ** -0.5
        attn_weights = torch.matmul(q, k.transpose(-2, -1)) * scale
        if key_padding_mask is not None:
            attn_weights = attn_weights.masked_fill(
                key_padding_mask.unsqueeze(1).unsqueeze(2), float('-inf')
            )
        attn_weights = torch.softmax(attn_weights, dim=-1)
        attn_weights = self.dropout(attn_weights)
        attn_output = torch.matmul(attn_weights, v)
        attn_output = attn_output.permute(0, 2, 1, 3).reshape(batch_size, tgt_len, self.d_model)
        return self.out_proj(attn_output)


class RoPEDecoderLayer(nn.Module):
    """Transformer decoder layer with proper RoPE in self-attention."""

    def __init__(self, d_model, n_heads, dim_feedforward=2048, dropout=0.1, max_seq_len=64):
        super().__init__()
        # Self-attention with RoPE
        self.self_attn = RoPEMultiHeadAttention(d_model, n_heads, dropout, max_seq_len)
        self.cross_attn = RoPECrossAttention(d_model, n_heads, dropout)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, tgt, memory, tgt_mask=None, memory_key_padding_mask=None, tgt_key_padding_mask=None):
        # Self-attention with RoPE
        tgt2 = self.self_attn(tgt, attn_mask=tgt_mask, key_padding_mask=tgt_key_padding_mask)
        tgt = tgt + self.dropout1(tgt2)
        tgt = self.norm1(tgt)
        # Cross-attention
        tgt2 = self.cross_attn(tgt, memory, key_padding_mask=memory_key_padding_mask)
        tgt = tgt + self.dropout2(tgt2)
        tgt = self.norm2(tgt)
        # Feedforward
        tgt2 = self.linear2(self.dropout(torch.relu(self.linear1(tgt))))
        tgt = tgt + self.dropout3(tgt2)
        tgt = self.norm3(tgt)

        return tgt


class ChronosTransformerDecoderRoPE(nn.Module):
    def __init__(self, embedding_size=4096,
                 d_model=256,
                 n_heads=4,
                 n_decoder_layers=4,
                 dim_feedforward=1024,
                 dropout=0.1,
                 prediction_length=64):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.n_decoder_layers = n_decoder_layers
        self.dim_feedforward = dim_feedforward
        self.dropout = dropout
        self.prediction_length = prediction_length
        self.embedding_size = embedding_size
        self.token_embedding = nn.Embedding(embedding_size, d_model)
        self.layers = nn.ModuleList([
            RoPEDecoderLayer(d_model, n_heads, dim_feedforward, dropout, prediction_length + 1)
            for _ in range(n_decoder_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.fc_out = nn.Linear(d_model, embedding_size)

    def forward(self, encoder_memory, tgt_tokens, tgt_padding_mask=None):
        x = self.token_embedding(tgt_tokens)
        # Create causal mask
        seq_len = tgt_tokens.size(1)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(seq_len, device=tgt_tokens.device)
        # Pass through decoder layers
        for layer in self.layers:
            x = layer(x, encoder_memory, tgt_mask=tgt_mask, tgt_key_padding_mask=tgt_padding_mask)
        x = self.norm(x)
        return self.fc_out(x)

    def generate(self, encoder_output, start_token=0, temperature=1.0, top_k=50):
        batch_size = encoder_output.size(0)
        device = encoder_output.device
        generated_tokens = torch.full((batch_size, 1), start_token, dtype=torch.long, device=device)
        for _ in range(self.prediction_length):
            logits = self.forward(encoder_output, generated_tokens)
            next_token_logits = logits[:, -1, :] / temperature
            if top_k > 0:
                top_k_logits, top_k_indices = torch.topk(next_token_logits, top_k, dim=-1)
                mask = torch.full_like(next_token_logits, float('-inf'))
                mask.scatter_(-1, top_k_indices, top_k_logits)
                next_token_logits = mask
            probs = torch.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            generated_tokens = torch.cat([generated_tokens, next_token], dim=1)
        return generated_tokens[:, 1:]

# Test ChronosTransformerDecoderRoPE
test_decoder = ChronosTransformerDecoderRoPE(
    embedding_size=4096,
    d_model=256,
    n_heads=4,
    n_decoder_layers=2,
    dim_feedforward=1024,
    dropout=0.1,
    prediction_length=32
)
test_tgt_tokens = torch.randint(0, 4096, (2, 1))  # Start with start token
decoder_output = test_decoder(test_encoder_output, test_tgt_tokens)
generated_tokens = test_decoder.generate(test_encoder_output, start_token=0, temperature=1.0, top_k=50)
print(f"Decoder Output shape: {decoder_output.shape}")

In [ ]:
# Let's understand the sampling.

vocab_size = 10
fake_logits = torch.tensor([[2.5, 1.8, 3.2, 1.0, 0.5, 0.3, 2.1, 1.5, 0.8, 0.2]])
token_names = [f"tok{i}" for i in range(vocab_size)]

print(f"Logits: {fake_logits[0].tolist()}")
print(f"Token with highest logit: {fake_logits.argmax().item()} ({fake_logits.max().item():.2f})")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Sampling', fontsize=16, fontweight='bold')


ax = axes[0]
colors = ['red' if i == fake_logits.argmax().item() else 'blue' for i in range(vocab_size)]
ax.bar(range(vocab_size), fake_logits[0].numpy(), color=colors, alpha=0.7)
ax.set_xlabel('Token ID')
ax.set_ylabel('Logit Value')
ax.set_title('Raw Logits from Model\n(Red = Greedy choice)')
ax.set_xticks(range(vocab_size))
ax.grid(True, alpha=0.3)


greedy_token = torch.argmax(fake_logits, dim=-1)
print(f"Selected token: {greedy_token.item()}")
greedy_sequence = []
for step in range(5):
    token = torch.argmax(fake_logits, dim=-1).item()
    greedy_sequence.append(token)
print(f"Generated sequence: {greedy_sequence}")


ax = axes[1]
temperatures = [0.5, 1.0, 2.0]
for temp in temperatures:
    scaled_logits = fake_logits / temp
    probs = torch.softmax(scaled_logits, dim=-1)[0].numpy()
    ax.plot(range(vocab_size), probs, marker='o', label=f'T={temp}', linewidth=2, markersize=8)

ax.set_xlabel('Token ID')
ax.set_ylabel('Probability')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xticks(range(vocab_size))

# Store probabilities for different temperatures
temp_probs = {}
for temp in [0.5, 1.0, 2.0]:
    print(f"\n   Temperature = {temp}")
    scaled_logits = fake_logits / temp
    probs = torch.softmax(scaled_logits, dim=-1)
    temp_probs[temp] = probs
    top_probs, top_idx = torch.topk(probs[0], k=3)
    for i, (prob, idx) in enumerate(zip(top_probs, top_idx)):
        print(f"      {i+1}. Token {idx.item()}: {prob.item():.4f} ({prob.item()*100:.1f}%)")
    sampled = []
    for _ in range(10):
        token = torch.multinomial(probs, num_samples=1).item()
        sampled.append(token)
    print(f"10 samples: {sampled}")
    print(f"Unique tokens: {len(set(sampled))}/10")

In [ ]:
# Create the full Simplified Chronos Model with RoPE

class SimplifiedChronosModelRoPE(nn.Module):

    def __init__(
        self,
        n_tokens=4096,
        d_model=256,
        nhead=4,
        num_encoder_layers=4,
        num_decoder_layers=4,
        dim_feedforward=1024,
        dropout=0.1,
        context_length=512,
        prediction_length=64,
    ):
        super().__init__()
        self.encoder = ChronosTransformerEncoderRoPE(
            embedding_size=n_tokens,
            d_model=d_model,
            n_heads=nhead,
            n_encoder_layers=num_encoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            context_length=context_length
        )
        self.decoder = ChronosTransformerDecoderRoPE(
            embedding_size=n_tokens,
            d_model=d_model,
            n_heads=nhead,
            n_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            prediction_length=prediction_length
        )
        # Shared token embedding
        self.shared_embedding = nn.Embedding(n_tokens, d_model)
        self.encoder.token_embedding = self.shared_embedding
        self.decoder.token_embedding = self.shared_embedding
        self.decoder.fc_out.weight = self.shared_embedding.weight

    def forward(self, src_tokens, tgt_tokens=None, src_padding_mask=None, tgt_padding_mask=None, temperature=1.0, top_k=20):
        encoder_output = self.encoder(src_tokens, src_padding_mask=src_padding_mask)
        if tgt_tokens is None:
            return self.decoder.generate(encoder_output, start_token=0, temperature=temperature, top_k=top_k)
        decoder_output = self.decoder(
            encoder_output,
            tgt_tokens,
            tgt_padding_mask=tgt_padding_mask
        )
        return decoder_output

    def predict(self, src_tokens, src_padding_mask=None, temperature=1.0, top_k=20):
        self.eval()
        with torch.no_grad():
            return self.forward(src_tokens, tgt_tokens=None, src_padding_mask=src_padding_mask, temperature=temperature, top_k=top_k)


# Create tiny RoPE model with same architecture as before
tiny_rope_model = SimplifiedChronosModelRoPE(
    n_tokens=4096,
    d_model=128,
    nhead=4,
    num_encoder_layers=3,
    num_decoder_layers=3,
    dim_feedforward=1024,
    dropout=0.1,
    context_length=128,
    prediction_length=24
)

# Count parameters
tiny_rope_params = sum(p.numel() for p in tiny_rope_model.parameters())
print(f"Tiny RoPE Model Parameters: {tiny_rope_params:,}")

In [ ]:
# Let's instantiate the official Chronos T5 Tiny model and count its parameters
from transformers import AutoModelForSeq2SeqLM

chronos_t5_tiny = AutoModelForSeq2SeqLM.from_pretrained("amazon/chronos-t5-tiny")
chronos_t5_tiny_params = sum(p.numel() for p in chronos_t5_tiny.parameters())
print(f"Chronos T5 Tiny Model: {chronos_t5_tiny_params:,} parameters")

In [ ]:
# Load training datasets

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from tqdm.auto import tqdm
import datasets

# ============================================================================
# TSMix - Real-world time series with TSMixup augmentation (10M samples)
# ============================================================================
train_tsmix_hf = datasets.load_dataset(
    "autogluon/chronos_datasets",
    "training_corpus_tsmixup_10m",
    streaming=True,
    split="train"
)

val_tsmix_hf = datasets.load_dataset(
    "autogluon/chronos_datasets",
    "training_corpus_tsmixup_10m",
    streaming=True,
    split="train"
)

# ============================================================================
# KernelSynth - Synthetic GP time series (1M samples)
# ============================================================================
train_kernelsynth_hf = datasets.load_dataset(
    "autogluon/chronos_datasets",
    "training_corpus_kernel_synth_1m",
    streaming=True,
    split="train"
)

val_kernelsynth_hf = datasets.load_dataset(
    "autogluon/chronos_datasets",
    "training_corpus_kernel_synth_1m",
    streaming=True,
    split="train"
)

In [ ]:
# Quick look at TSMix training data

sample_tsmix = next(iter(train_tsmix_hf))
print(f"TSMix Keys: {list(sample_tsmix.keys())}")

# Visualize one sample
plt.figure(figsize=(12, 4))
plt.plot(sample_tsmix['target'][:200])
plt.title('Sample Time Series from TSMix (Real-World Data with Augmentation)')
plt.xlabel('Time Steps')
plt.ylabel('Value')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Quick look at KernelSynth training data

sample_kernelsynth = next(iter(train_kernelsynth_hf))
print(f"KernelSynth Keys: {list(sample_kernelsynth.keys())}")

# Visualize one sample
plt.figure(figsize=(12, 4))
plt.plot(sample_kernelsynth['target'][:200])
plt.xlabel('Time Steps')
plt.ylabel('Value')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
!pip install gluonts

In [ ]:
from torch.utils.data import IterableDataset, get_worker_info
from typing import Optional, Iterator

from gluonts.transform import (
    FilterTransformation,
    TestSplitSampler,
    ValidationSplitSampler,
    InstanceSplitter,
    ExpectedNumInstanceSampler,
    MissingValueImputation,
    LeavesMissingValues,
    LastValueImputation,
)

from gluonts.itertools import Cyclic, Map, Filter

from functools import partial
from typing import List, Iterator, Optional, Dict

import itertools


In [ ]:
# The chronos framework does not expose training classes, so here is the redacted copy.
# You can checkout open-source implementation and import training classes from
# there.
class ChronosDataset(IterableDataset):

    def __init__(
        self,
        datasets: list,
        probabilities: list[float],
        tokenizer: ChronosTokenizer,
        context_length: int = 512,
        prediction_length: int = 64,
        drop_prob: float = 0.2,
        min_past: Optional[int] = None,
        model_type: str = "seq2seq",
        imputation_method: Optional[MissingValueImputation] = None,
        mode: str = "training",
        np_dtype=np.float32,
    ) -> None:
        super().__init__()

        assert len(probabilities) == len(datasets)
        assert mode in ("training", "validation", "test")
        assert model_type in ("seq2seq", "causal")

        self.datasets = datasets
        self.probabilities = probabilities
        self.tokenizer = tokenizer
        self.context_length = context_length
        self.prediction_length = prediction_length
        self.drop_prob = drop_prob if model_type == "seq2seq" else 0.0
        self.min_past = min_past or prediction_length
        self.model_type = model_type
        self.imputation_method = None
        self.mode = mode
        self.np_dtype = np_dtype

    def preprocess_entry(self, entry: dict, mode: str) -> dict:
        entry = {f: entry[f] for f in ["start", "target"]}
        entry["target"] = np.asarray(entry["target"], dtype=self.np_dtype)
        assert entry["target"].ndim == 1, f"got {entry['target'].ndim=}, expected 1"

        if self.model_type == "causal":
            # Causal models do not play nice with missing values, so it is
            # recommended to use an imputation method, e.g., LastValueImputation
            entry["target"] = self.imputation_method(entry["target"])

        if mode == "training" and self.drop_prob > 0:
            target = entry["target"].copy()
            drop_p = np.random.uniform(low=0.0, high=self.drop_prob)
            mask = np.random.choice(
                [True, False], size=len(target), p=[drop_p, 1 - drop_p]
            )
            target[mask] = np.nan
            entry["target"] = target

        return entry

    def _create_instance_splitter(self, mode: str):
        assert mode in ["training", "test", "validation"]

        instance_sampler = {
            "training": ExpectedNumInstanceSampler(
                num_instances=1.0,
                min_instances=1,
                min_past=self.min_past,
                min_future=self.prediction_length,
            ),
            "test": TestSplitSampler(),
            "validation": ValidationSplitSampler(min_future=self.prediction_length),
        }[mode]

        return InstanceSplitter(
            target_field="target",
            is_pad_field="is_pad",
            start_field="start",
            forecast_start_field="forecast_start",
            instance_sampler=instance_sampler,
            past_length=self.context_length,
            future_length=self.prediction_length,
            dummy_value=np.nan,
        )

    def create_training_data(self, data):
        data = Cyclic(data)
        split_transform = self._create_instance_splitter(
            "training"
        ) + FilterTransformation(
            condition=lambda entry: (~np.isnan(entry["past_target"])).sum() > 0
        )
        data = split_transform.apply(data, is_train=True)
        return data

    def create_test_data(self, data):
        data = self._create_instance_splitter("test").apply(data, is_train=False)
        return data

    def create_validation_data(self, data):
        data = self._create_instance_splitter("validation").apply(data, is_train=False)
        return data

    def to_hf_format(self, entry: dict) -> dict:
        past_target = torch.tensor(entry["past_target"]).unsqueeze(0)
        input_ids, attention_mask, scale = self.tokenizer.context_input_transform(
            past_target
        )
        future_target = torch.tensor(entry["future_target"]).unsqueeze(0)
        labels, labels_mask = self.tokenizer.label_input_transform(future_target, scale)
        labels[labels_mask == 0] = -100

        if self.model_type == "causal":
            # The InstanceSplitter pads time series on the left to be equal to the
            # context_length. However, certain models (e.g., GPT2) with absolute
            # position embeddings should not be trained with left padding.
            # The following piece of code moves padding from left to right.

            assert input_ids.shape[-1] == entry["past_is_pad"].shape[0]

            # Find the index where padding starts
            pad_start_idx = np.searchsorted(1 - entry["past_is_pad"], 1)
            padded_input_ids, obs_input_ids = torch.tensor_split(
                input_ids, [pad_start_idx], dim=-1
            )
            padded_attention_mask, obs_attention_mask = torch.tensor_split(
                attention_mask, [pad_start_idx], dim=-1
            )

            # Move padding to the right
            input_ids = torch.cat(
                [
                    obs_input_ids,
                    labels,
                    padded_input_ids,
                ],
                axis=-1,
            )
            attention_mask = torch.cat(
                [
                    obs_attention_mask,
                    labels_mask,
                    padded_attention_mask,
                ],
                axis=-1,
            )

            # labels for causal models are same as the input_ids.
            # Internally transformers shifts the labels by one during training.
            labels = input_ids.clone()
            input_ids[~attention_mask] = self.tokenizer.config.pad_token_id
            labels[~attention_mask] = -100

        return {
            "input_ids": input_ids.squeeze(0),
            "attention_mask": attention_mask.squeeze(0),
            "labels": labels.squeeze(0),
        }

    def __iter__(self) -> Iterator:
        preprocessed_datasets = [
            Map(
                partial(self.preprocess_entry, mode=self.mode),
                dataset,
            )
            for dataset in self.datasets
        ]

        if self.mode == "training":
            iterables = [
                self.create_training_data(dataset) for dataset in preprocessed_datasets
            ]
        elif self.mode == "test":
            iterables = [
                self.create_test_data(dataset) for dataset in preprocessed_datasets
            ]
        else:
            iterables = [
                self.create_validation_data(dataset)
                for dataset in preprocessed_datasets
            ]

        worker_info = get_worker_info()
        if worker_info is None:
            probs = list(self.probabilities)
        else:
            worker_id = worker_info.id
            num_workers = worker_info.num_workers
            iterables = list(itertools.islice(iterables, worker_id, None, num_workers))
            probs = list(
                itertools.islice(self.probabilities, worker_id, None, num_workers)
            )

        probs = [prob / sum(probs) for prob in probs]

        iterators = list(map(iter, iterables))
        if self.mode == "training":
            while True:
                idx = np.random.choice(range(len(iterators)), p=probs)
                try:
                    yield self.to_hf_format(next(iterators[idx]))
                except StopIteration:
                    probs[idx] = 0
                    if sum(probs) == 0:
                        return
                    probs = [prob / sum(probs) for prob in probs]
        else:
            for entry in itertools.chain(*iterators):
                yield self.to_hf_format(entry)


In [ ]:


import pandas as pd
from tqdm.auto import tqdm

# Convert HuggingFace streaming dataset to GluonTS format
class HFDatasetWrapper:
    """Wrapper to convert HuggingFace streaming dataset to GluonTS format."""
    def __init__(self, hf_stream, num_samples=5000, name="Dataset"):
        self.data = []
        for i, sample in enumerate(tqdm(hf_stream, total=num_samples, desc=f"Loading {name}")):
            if i >= num_samples:
                break
            if len(sample['target']) >= 200:
                self.data.append({
                    'start': pd.Period('2000-01-01', freq='h'),
                    'target': np.array(sample['target'], dtype=np.float32)
                })

    def __iter__(self):
        return iter(self.data)

    def __len__(self):
        return len(self.data)

# TSMix samples
tsmix_train_samples = 500
tsmix_val_samples = 50

train_data_tsmix = HFDatasetWrapper(train_tsmix_hf, num_samples=tsmix_train_samples, name="TSMix Train")
val_data_tsmix = HFDatasetWrapper(val_tsmix_hf, num_samples=tsmix_val_samples, name="TSMix Val")

# KernelSynth samples
kernelsynth_train_samples = 500
kernelsynth_val_samples = 50

train_data_kernelsynth = HFDatasetWrapper(train_kernelsynth_hf, num_samples=kernelsynth_train_samples, name="KernelSynth Train")
val_data_kernelsynth = HFDatasetWrapper(val_kernelsynth_hf, num_samples=kernelsynth_val_samples, name="KernelSynth Val")

# Create tokenzier config and dataset
from chronos import ChronosConfig
from dataclasses import dataclass

@dataclass
class SimpleChronosConfig(ChronosConfig):
    tokenizer_class: str = "ChronosTokenizer"
    tokenizer_kwargs: dict = None
    context_length: int = 128
    prediction_length: int = 24
    n_tokens: int = 4096
    n_special_tokens: int = 2
    pad_token_id: int = 0
    eos_token_id: int = 1
    use_eos_token: bool = True
    model_type: str = "seq2seq"
    num_samples: int = 20
    temperature: float = 1.0
    top_k: int = 50
    top_p: float = 1.0

    def __post_init__(self):
        if self.tokenizer_kwargs is None:
            self.tokenizer_kwargs = {"low_limit": -15.0, "high_limit": 15.0}
        super().__post_init__()

config = SimpleChronosConfig()

# Use our custom ChronosTokenizer instead of MeanScaleUniformBins
official_tokenizer = ChronosTokenizer(
    low_limit=config.tokenizer_kwargs["low_limit"],
    high_limit=config.tokenizer_kwargs["high_limit"],
    config=config
)

# Use both datasets with equal probability (50% each)
train_dataset = ChronosDataset(
    datasets=[train_data_tsmix, train_data_kernelsynth],
    probabilities=[0.5, 0.5],  # 50% TSMix, 50% KernelSynth
    tokenizer=official_tokenizer,
    context_length=128,
    prediction_length=24,
    min_past=60,
    model_type="seq2seq",
    mode="training",
)

val_dataset = ChronosDataset(
    datasets=[val_data_tsmix, val_data_kernelsynth],
    probabilities=[0.5, 0.5],
    tokenizer=official_tokenizer,
    context_length=128,
    prediction_length=24,
    min_past=60,
    model_type="seq2seq",
    mode="validation",
)

print(f"Training Data:")
print(f"- TSMix (real):        {len(train_data_tsmix):,} samples")
print(f"- KernelSynth (synth): {len(train_data_kernelsynth):,} samples")
print(f"- Total:               {len(train_data_tsmix) + len(train_data_kernelsynth):,} samples")

print(f"Validation Data:")
print(f"- TSMix (real):        {len(val_data_tsmix):,} samples")
print(f"- KernelSynth (synth): {len(val_data_kernelsynth):,} samples")
print(f"- Total:               {len(val_data_tsmix) + len(val_data_kernelsynth):,} samples")

In [ ]:
# Set up training
device = torch.device("cuda") # Use MPS if you have Apple Silicon
print(f"Using device: {device}")

tiny_rope_model = tiny_rope_model.to(device)

# Create data loaders (no shuffle for IterableDataset)
train_loader = DataLoader(train_dataset, batch_size=64, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=64, num_workers=0)

# Create optimizer and criterion
tiny_rope_optimizer = AdamW(tiny_rope_model.parameters(), lr=0.001, weight_decay=0.01)
tiny_rope_criterion = nn.CrossEntropyLoss(ignore_index=-100)

In [ ]:
# Common training loop function for both pre-training and fine-tuning

def train_model(
    model,
    train_loader,
    optimizer,
    criterion,
    device,
    num_epochs,
    batches_per_epoch,
    val_loader=None,
    val_batches=50,
    title="Training",
    plot_color='blue'
):
    train_losses = []
    val_losses = []
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        epoch_train_loss = 0
        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")
        for batch_idx, batch in enumerate(train_pbar):
            if batch_idx >= batches_per_epoch:
                break
            src_tokens = batch['input_ids'].to(device)
            tgt_tokens = batch['labels'].to(device)
            # Forward pass
            optimizer.zero_grad()
            # Prepend start token to match T5's _shift_right() behavior:
            # decoder_input: [START, label[0], label[1], ...]
            # labels/target: [label[0], label[1], ..., EOS]
            batch_size = tgt_tokens.shape[0]
            start_tokens = torch.zeros((batch_size, 1), dtype=tgt_tokens.dtype, device=device)
            decoder_input = torch.cat([start_tokens, tgt_tokens[:, :-1].clone()], dim=1)
            # Replace -100 (ignore_index for loss) with 0 (pad token) for valid
            # embedding lookup. Look at:
            # https://github.com/huggingface/transformers/blob/main/src/transformers/models/t5/modeling_t5.py#L596
            # Where the _shift_right() shift the input right (by appending the
            # start token) and filling all -100 with zeros.
            decoder_input[decoder_input == -100] = 0
            logits = model(src_tokens, decoder_input)
            loss = criterion(
                logits.reshape(-1, logits.size(-1)),
                tgt_tokens.reshape(-1)  # Full labels now, not tgt_tokens[:, 1:]
            )
            # Backward pass
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            # Loss tracking
            epoch_train_loss += loss.item()
            train_pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        avg_train_loss = epoch_train_loss / (batch_idx + 1)
        train_losses.append(avg_train_loss)
        # Validation phase (if val_loader provided)
        if val_loader is not None:
            model.eval()
            epoch_val_loss = 0
            val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]")
            with torch.no_grad():
                for batch_idx, batch in enumerate(val_pbar):
                    if batch_idx >= val_batches:
                        break
                    src_tokens = batch['input_ids'].to(device)
                    tgt_tokens = batch['labels'].to(device)
                    # Prepend start token for validation as well
                    batch_size = tgt_tokens.shape[0]
                    start_tokens = torch.zeros((batch_size, 1), dtype=tgt_tokens.dtype, device=device)
                    decoder_input = torch.cat([start_tokens, tgt_tokens[:, :-1].clone()], dim=1)
                    decoder_input[decoder_input == -100] = 0
                    logits = model(src_tokens, decoder_input)
                    loss = criterion(
                        logits.reshape(-1, logits.size(-1)),
                        tgt_tokens.reshape(-1)
                    )
                    epoch_val_loss += loss.item()
                    val_pbar.set_postfix({'loss': f'{loss.item():.4f}'})
            # Loss tracking
            avg_val_loss = epoch_val_loss / (batch_idx + 1)
            val_losses.append(avg_val_loss)
            print(f"\nEpoch {epoch+1}/{num_epochs} - Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}\n")
        else:
            print(f"Epoch {epoch+1}/{num_epochs} - Loss: {avg_train_loss:.4f}")

    # Plot training curves
    plt.figure(figsize=(10, 5))
    plt.plot(range(1, num_epochs+1), train_losses, marker='o', label='Train Loss', color=plot_color, linewidth=2)
    if val_losses:
        plt.plot(range(1, num_epochs+1), val_losses, marker='s', label='Val Loss', color='orange', linewidth=2)
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(f'{title} Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    return train_losses, val_losses

In [ ]:
# Common evaluation function for testing model on airline passengers

def evaluate_model_on_airline(
    model,
    context,
    target,
    tokenizer,
    device,
    num_samples=100,
    temperature=1.0,
    top_k=50,
    title="Model Forecast",
    plot_color='green'
):
    # Pad context to 128 if needed (model expects this length)
    if len(context) < 128:
        padded_context = np.concatenate([np.full(128 - len(context), np.nan), context])
    else:
        padded_context = context[-128:]

    # Convert to torch tensor and add batch dimension
    context_tensor = torch.tensor(padded_context, dtype=torch.float32).unsqueeze(0)
    # Tokenize using official tokenizer's context_input_transform method
    token_ids, attention_mask, scale = tokenizer.context_input_transform(context_tensor)

    print(f"Debug: Context shape={context.shape}, Scale={scale.item():.2f}, Context mean={np.mean(context):.2f}")

    # The official tokenizer adds EOS token making sequence 129, but our encoder only has 128 positions
    if token_ids.shape[1] > 128:
        token_ids = token_ids[:, :128]
        attention_mask = attention_mask[:, :128]
    test_tokens_tensor = token_ids.to(device)
    # Generate MULTIPLE predictions for probabilistic forecasting
    all_forecast_samples = []
    model.eval()
    with torch.no_grad():
        for sample_idx in range(num_samples):
            predicted_tokens = model.predict(test_tokens_tensor, temperature=temperature, top_k=top_k)
            # Remove EOS token (token 1) if present at the end
            predicted_tokens_clean = predicted_tokens.clone()
            if predicted_tokens_clean[0, -1].item() == 1:
                predicted_tokens_clean = predicted_tokens_clean[:, :-1]
            # Move to CPU and add sample dimension for output_transform
            predicted_tokens_cpu = predicted_tokens_clean.cpu().unsqueeze(1)
            predicted_values_tensor = tokenizer.output_transform(predicted_tokens_cpu, scale)
            # Extract the values and convert to numpy
            predicted_values = predicted_values_tensor.squeeze().numpy()
            # Pad to target length if needed (in case EOS was removed)
            if len(predicted_values) < len(target):
                predicted_values = np.pad(predicted_values, (0, len(target) - len(predicted_values)), mode='edge')
            all_forecast_samples.append(predicted_values)
    # Convert to array: (num_samples, prediction_length)
    forecast_samples = np.array(all_forecast_samples)
    # Calculate statistics across samples
    forecast_median = np.median(forecast_samples, axis=0)
    forecast_10p = np.percentile(forecast_samples, 10, axis=0)
    forecast_90p = np.percentile(forecast_samples, 90, axis=0)
    # Visualize the results with prediction intervals
    plt.figure(figsize=(14, 6))
    # Plot historical context
    plt.plot(range(len(context)), context, label='Context (Historical Data)', color='blue', linewidth=2)
    # Plot actual target
    plt.plot(range(len(context), len(context) + len(target)), target,
             label='Actual Target (Future Data)', color='red', linewidth=2)
    # Plot model's forecast median
    plt.plot(range(len(context), len(context) + len(target)), forecast_median,
             label='Forecast Median', color=plot_color, linewidth=2, linestyle='--')
    # Fill between 10th and 90th percentiles for prediction interval
    plt.fill_between(
        range(len(context), len(context) + len(target)),
        forecast_10p,
        forecast_90p,
        color=plot_color,
        alpha=0.3,
        label='80% Prediction Interval (10th-90th Percentile)'
    )
    plt.axvline(x=len(context), color='black', linestyle='--', linewidth=1.5, alpha=0.5, label='Forecast Start')
    plt.xlabel('Months')
    plt.ylabel('Number of Passengers')
    plt.title(f'{title} ({num_samples} Samples)')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

    # Calculate error metrics using median
    mae = np.mean(np.abs(target - forecast_median))
    mape = np.mean(np.abs((target - forecast_median) / target)) * 100
    rmse = np.sqrt(np.mean((target - forecast_median)**2))

    print(f"Forecast Accuracy (using median of {num_samples} samples):")
    print(f"- MAE (Mean Absolute Error): {mae:.2f} passengers")
    print(f"- MAPE (Mean Absolute Percentage Error): {mape:.2f}%")
    print(f"- RMSE (Root Mean Squared Error): {rmse:.2f} passengers")

    # Show some sample diversity statistics
    interval_width = forecast_90p - forecast_10p
    print(f"Forecast Uncertainty:")
    print(f"- Mean prediction interval width: {np.mean(interval_width):.2f} passengers")
    print(f"- Max prediction interval width: {np.max(interval_width):.2f} passengers")
    print(f"- Min prediction interval width: {np.min(interval_width):.2f} passengers")

    # Debug: Show token distribution
    print(f"Debug: Sample token range: {predicted_tokens_clean.min().item()} to {predicted_tokens_clean.max().item()}")
    print(f"Debug: Forecast median range: {forecast_median.min():.2f} to {forecast_median.max():.2f}")

    return {
        'forecast_samples': forecast_samples,
        'forecast_median': forecast_median,
        'forecast_10p': forecast_10p,
        'forecast_90p': forecast_90p,
        'mae': mae,
        'mape': mape,
        'rmse': rmse
    }

In [ ]:
# Train the RoPE model using the common training loop

tiny_rope_train_losses, tiny_rope_val_losses = train_model(
    model=tiny_rope_model,
    train_loader=train_loader,
    optimizer=tiny_rope_optimizer,
    criterion=tiny_rope_criterion,
    device=device,
    num_epochs=10,
    batches_per_epoch=150,
    val_loader=val_loader,
    val_batches=5,
    title="Tiny RoPE Model Training and Validation",
    plot_color='blue'
)

In [ ]:
# Test our trained model on the airline passengers example.

pretrained_results = evaluate_model_on_airline(
    model=tiny_rope_model,
    context=context,
    target=target,
    tokenizer=official_tokenizer,
    device=device,
    num_samples=100,
    temperature=1.0,
    top_k=50,
    title="Airline Passengers Forecast - Pre-trained Model",
    plot_color='green'
)

# Store metrics for comparison later
mae = pretrained_results['mae']
mape = pretrained_results['mape']
rmse = pretrained_results['rmse']
forecast_median = pretrained_results['forecast_median']
forecast_10p = pretrained_results['forecast_10p']
forecast_90p = pretrained_results['forecast_90p']

In [ ]:
# Create airline-specific training samples using sliding windows WITH AUGMENTATION
# Adding noise and scaling variations to prevent memorization

class AirlineDatasetWrapper:
    """Create multiple training samples from the airline dataset with augmentation."""
    def __init__(self, data, context_length=64, prediction_length=24, stride=1,
                 num_augmentations=10, noise_std=0.05, scale_range=(0.8, 1.2)):
        self.data = []
        # Slide through the data creating training samples
        for i in range(0, len(data) - context_length - prediction_length, stride):
            context_end = i + context_length
            pred_end = context_end + prediction_length
            full_series = data[i:pred_end].copy()
            # Original sample (no augmentation)
            self.data.append({
                'start': pd.Period('2000-01-01', freq='M'),
                'target': np.array(full_series, dtype=np.float32)
            })
            # Create augmented versions
            local_std = np.std(full_series)
            for aug_idx in range(num_augmentations):
                augmented_series = full_series.copy()
                # 1. Add Gaussian noise (proportional to local variation)
                noise = np.random.normal(0, noise_std * local_std, len(augmented_series))
                augmented_series = augmented_series + noise
                # 2. Random scaling (simulates different passenger volumes)
                scale_factor = np.random.uniform(scale_range[0], scale_range[1])
                augmented_series = augmented_series * scale_factor
                # 3. Small trend shift (random walk component)
                if np.random.random() > 0.5:
                    trend_shift = np.cumsum(np.random.normal(0, 0.5, len(augmented_series)))
                    augmented_series = augmented_series + trend_shift
                self.data.append({
                    'start': pd.Period('2000-01-01', freq='M'),
                    'target': np.array(augmented_series, dtype=np.float32)
                })

        print(f"Created {len(self.data)} training samples from airline dataset")
        print(f"- Original windows: {len(self.data) // (num_augmentations + 1)}")
        print(f"- Augmentations per window: {num_augmentations}")
        print(f"- Noise std: {noise_std * 100:.1f}% of local std")
        print(f"- Scale range: {scale_range}")

    def __iter__(self):
        return iter(self.data)

    def __len__(self):
        return len(self.data)

airline_data = df['Passengers'].values.astype(float)
airline_train_data = AirlineDatasetWrapper(
    airline_data,
    context_length=64,
    prediction_length=24,
    stride=1,
    num_augmentations=10,
    noise_std=0.05,
    scale_range=(0.7, 1.3)
)

airline_finetune_dataset = ChronosDataset(
    datasets=[airline_train_data],
    probabilities=[1.0],
    tokenizer=official_tokenizer,
    context_length=128,
    prediction_length=24,
    min_past=60,
    model_type="seq2seq",
    mode="training",
)

print(f"- Total samples: {len(airline_train_data)}")

In [ ]:
# Fine-tune the pre-trained model on airline dataset

# Create dataloader for fine-tuning
finetune_loader = DataLoader(airline_finetune_dataset, batch_size=8, num_workers=0)

# Use lower learning rate for fine-tuning (to not destroy pre-trained knowledge)
finetune_optimizer = AdamW(tiny_rope_model.parameters(), lr=0.0001, weight_decay=0.01)

# Fine-tune using the common training loop
finetune_losses, _ = train_model(
    model=tiny_rope_model,
    train_loader=finetune_loader,
    optimizer=finetune_optimizer,
    criterion=tiny_rope_criterion,
    device=device,
    num_epochs=20,
    batches_per_epoch=50,
    val_loader=None,  # No validation for fine-tuning
    title="Fine-Tuning on Airline Dataset",
    plot_color='orange'
)

In [ ]:
# Test the fine-tuned model on the airline passengers forecast

finetuned_results = evaluate_model_on_airline(
    model=tiny_rope_model,
    context=context,
    target=target,
    tokenizer=official_tokenizer,
    device=device,
    num_samples=100,
    temperature=1.0,
    top_k=50,
    title="Airline Passengers Forecast - Fine-Tuned Model",
    plot_color='purple'
)

# Store metrics for comparison
mae_ft = finetuned_results['mae']
mape_ft = finetuned_results['mape']
rmse_ft = finetuned_results['rmse']
forecast_median_ft = finetuned_results['forecast_median']
forecast_10p_ft = finetuned_results['forecast_10p']
forecast_90p_ft = finetuned_results['forecast_90p']